In [2]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install pyro-ppl seaborn

Note: you may need to restart the kernel to use updated packages.


In [4]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from devinterp.optim.sgld import SGLD
from devinterp.optim.sgnht import SGNHT
from devinterp.slt import estimate_learning_coeff_with_summary, sample
from devinterp.slt.llc import LLCEstimator, SamplerCallback, OnlineLLCEstimator
from devinterp.utils import plot_trace

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
sns.set_style("whitegrid")

# plotting
CMAP = sns.color_palette("muted", as_cmap=True)
PRIMARY, SECONDARY, TERTIARY = CMAP[:3]

/var/folders/ds/qvg7h1s93x16vvczwp8vr83m0000gn/T/ipykernel_15874/2903749014.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# constants
SIGMA = 0.25
NUM_TRAIN_SAMPLES = 1000
BATCH_SIZE = NUM_TRAIN_SAMPLES
CRITERION = F.mse_loss

In [6]:
class SingularModel(nn.Module):
    def __init__(self, powers):
        super(SingularModel, self).__init__()
        self.weights = nn.Parameter(
            torch.tensor([0.0, 0.0], dtype=torch.float32, requires_grad=True).to(DEVICE)
        )
        self.powers = powers

    def forward(self, x):
        multiplied = torch.prod(self.weights**self.powers)
        x = x * multiplied
        return x

def generate_dataset_for_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    x = torch.normal(0, 2, size=(NUM_TRAIN_SAMPLES,))
    y = SIGMA * torch.normal(0, 1, size=(NUM_TRAIN_SAMPLES,))
    train_data = TensorDataset(x, y)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    return train_loader, train_data, x, y

In [7]:
train_loader, train_data, x, y = generate_dataset_for_seed(0)
lr = 0.0001
num_chains = 3
num_draws = 5000
powers_to_test = [
    torch.tensor([0, 2]).to(DEVICE), # minimally singular
    torch.tensor([2, 0]).to(DEVICE) # minimally sigular
]
rlct_estimates_sgld = {
    tuple(power.detach().cpu().tolist()): estimate_learning_coeff_with_summary(
        model=SingularModel(power),
        loader=train_loader,
        optimizer_kwargs=dict(
            lr=lr,
            bounding_box_size=1.0,
        ),
        criterion=CRITERION,
        sampling_method=SGLD,
        num_chains=num_chains,
        num_draws=num_draws,
        device=DEVICE,
    )
    for power in powers_to_test
}
rlct_estimates_sgnht = {
    tuple(power.detach().cpu().tolist()): estimate_learning_coeff_with_summary(
        model=SingularModel(power),
        loader=train_loader,
        optimizer_kwargs=dict(
            lr=lr,
            bounding_box_size=1.0,
            diffusion_factor=0.01,
        ),
        criterion=CRITERION,
        sampling_method=SGNHT,
        num_chains=num_chains,
        num_draws=num_draws,
        device=DEVICE,
    )
    for power in powers_to_test
}

/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/devinterp/slt/sampler.py:170: UserWarning: You are taking more draws than burn-in steps, your LLC estimates will likely be underestimates. Please check LLC chain convergence.
  warnings.warn(
/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/devinterp/slt/sampler.py:174: UserWarning: You are taking more sample batches than there are dataloader batches available, this removes some randomness from sampling but is probably fine. (All sample batches beyond the number dataloader batches are cycled from the start, f.e. 9 samples from [A, B, C] would be [B, A, C, B, A, C, B, A, C].)
  warnings.warn(
/Users/ragharao/opt/anaconda3/envs/devinterp_env/lib/python3.11/site-packages/devinterp/slt/sampler.py:58: UserWarning: You are taking more sample batches than there are dataloader batches available, this removes some randomness from sampling but is probably fine. (All sample batches beyond th

In [8]:
for sampler_type, estimates in zip(
    ("sgld", "sgnht"), (rlct_estimates_sgld, rlct_estimates_sgnht)
):
    df_data = {
        "powers": [k for k in estimates.keys()],
        "LLC_Summary": estimates.values(),
    }
    df = pd.DataFrame(df_data)
    df["llc_mean"] = df["LLC_Summary"].apply(lambda x: x["llc/mean"])
    df["llc_std"] = df["LLC_Summary"].apply(lambda x: x["llc/std"])
    df = df.drop("LLC_Summary", axis=1)
    print(sampler_type)
    print(df.to_string(index=False))

sgld
powers  llc_mean  llc_std
(0, 2)  0.235147 0.050897
(2, 0)  0.177827 0.032972
sgnht
powers  llc_mean  llc_std
(0, 2)  0.279457 0.024806
(2, 0)  0.281580 0.024980


In [15]:
def sum(n):
    if n == 1:
        return 1
    return n + sum(n-1)

for i in range(1000):
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [ ]:
hessians = produce_hessians(models=models,
                                data_loader=train_loader,
                                num_batches=args.hessian_batch_size,
                                criterion=metric,
                                device=device,
                                history=True)